# 🏁 Kaggle HBAAC 2026 — Demand Forecasting v4
**Approach:** Global LightGBM + comprehensive feature engineering + smart post-processing.

**What's new in v4 (vs v3 baseline WRMSSE=0.5585):**
- ✅ Yearly seasonality lags: `lag_357`, `lag_365`, `lag_371`
- ✅ Exponential weighted mean features: `ewm_mean_28`, `ewm_mean_56`
- ✅ Tet season flags: `is_tet_season`, `days_to_tet`, `days_since_tet`
- ✅ Extended holiday windows: pre-14d, post-7d
- ✅ Extra SKU stats: `sku_median_qty`, `sku_cv`, `sku_active_days`
- ✅ Calendar: `days_in_month`, `is_month_start`, `is_month_end`
- ✅ Tuned LightGBM: `num_leaves=255`, `lr=0.05`, `early_stop=100`
- ✅ Profit-aware sample weights (power 0.7, capped at 50× median)
- ✅ Profit-aware post-processing tiers (replaces mean_qty-based tiers)
- ✅ Fixed inference: pre-computed `hist_pivot` (no O(N²) rebuild per day)
- ✅ Sparse SKU filter by absolute `active_days < 5`

---
## 📦 Cell 1 — Libraries & Constants

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

INPUT_DIR  = Path('/kaggle/input/competitions/hbaac-round2')
OUTPUT_DIR = Path('/kaggle/working')
TRAIN_PATH  = INPUT_DIR / 'train.csv'
SAMPLE_PATH = INPUT_DIR / 'sample_submission.csv'

TRAIN_START = pd.Timestamp('2020-11-17')
TRAIN_END   = pd.Timestamp('2025-09-05')
VAL_START   = pd.Timestamp('2025-09-06')
VAL_END     = pd.Timestamp('2025-10-03')
EVAL_START  = pd.Timestamp('2025-10-04')
EVAL_END    = pd.Timestamp('2025-10-31')
SPLIT_DATE  = pd.Timestamp('2025-07-12')
HORIZON     = 56
SEED        = 42
print('Libraries loaded \u2713')

---
## 🗂️ Cell 2 — Load & Parse Raw Data

In [ ]:
def parse_vnd(series: pd.Series) -> pd.Series:
    """Parse Vietnamese decimal-comma number strings to float."""
    return (
        series.astype(str).str.strip()
        .str.replace('.', '', regex=False)
        .str.replace(',', '.', regex=False)
        .replace({'nan': np.nan, '': np.nan})
        .astype(float)
    )

print('Reading train.csv ...')
raw = pd.read_csv(
    TRAIN_PATH,
    dtype={'Stt': 'str', 'ItemCode': 'str', 'UnitPrice': 'str', 'Unit Cost': 'str'},
    parse_dates=['Date'],
    low_memory=False
)
raw['Quantity']    = pd.to_numeric(raw['Quantity'],    errors='coerce').fillna(0).astype('int32')
raw['SalesAmount'] = pd.to_numeric(raw['SalesAmount'], errors='coerce').fillna(0).astype('int64')
raw['Cost Amount'] = pd.to_numeric(raw['Cost Amount'], errors='coerce').fillna(0).astype('int64')
raw['UnitPrice'] = parse_vnd(raw['UnitPrice'])
raw['Unit Cost'] = parse_vnd(raw['Unit Cost'])
raw['ItemCode'] = raw['ItemCode'].astype('category')

print(f'Raw rows      : {len(raw):,}')
print(f'SKUs          : {raw["ItemCode"].nunique():,}')
print(f'Date range    : {raw["Date"].min().date()} \u2192 {raw["Date"].max().date()}')
print(f'Returns rows  : {(raw["Quantity"] < 0).sum():,}  ({(raw["Quantity"] < 0).mean()*100:.1f}%)')

---
## 🔍 Cell 3 — Per-SKU Profit Weights

In [ ]:
raw['Profit'] = raw['SalesAmount'].astype(float) - raw['Cost Amount'].astype(float)
sku_profit = (
    raw.groupby('ItemCode', observed=True)['Profit']
       .sum().reset_index().rename(columns={'Profit': 'total_profit'})
)
sku_profit['total_profit'] = sku_profit['total_profit'].clip(lower=0)
total = sku_profit['total_profit'].sum()
sku_profit['weight'] = sku_profit['total_profit'] / total if total > 0 else 0.0
print(f'SKUs with positive profit : {(sku_profit["total_profit"] > 0).sum():,}')
print(f'Total profit (VND)        : {total:,.0f}')
sku_profit.nlargest(10, 'weight')[['ItemCode', 'total_profit', 'weight']]

---
## 🧹 Cell 4 — Handle Returns & Aggregate Daily

In [ ]:
daily = (
    raw.groupby(['ItemCode', 'Date'], observed=True)
    .agg(
        Quantity=('Quantity', 'sum'),
        SalesAmount=('SalesAmount', 'sum'),
        CostAmount=('Cost Amount', 'sum'),
    )
    .reset_index()
)
daily['Quantity'] = daily['Quantity'].clip(lower=0)
print(f'Daily grain rows: {len(daily):,}')

---
## 📅 Cell 5 — Reindex to Full Daily Calendar

In [ ]:
all_dates = pd.date_range(TRAIN_START, TRAIN_END, freq='D')
all_skus  = daily['ItemCode'].cat.categories if hasattr(daily['ItemCode'], 'cat') \
            else daily['ItemCode'].unique()
full_idx = pd.MultiIndex.from_product([all_skus, all_dates], names=['ItemCode', 'Date'])
print(f'Full grid: {len(full_idx):,} rows ({len(all_skus):,} SKUs \u00d7 {len(all_dates):,} days)')

daily = (
    daily.set_index(['ItemCode', 'Date'])
    .reindex(full_idx, fill_value=0)
    .reset_index()
)
daily['ItemCode'] = daily['ItemCode'].astype('category')
daily['Quantity'] = daily['Quantity'].astype('float32')
print(f'After reindex: {len(daily):,} rows')

---
## 🗓️ Cell 6 — Vietnamese Public Holidays (Extended)

In [ ]:
def get_vietnamese_holidays(start_year=2020, end_year=2026):
    """Returns (DatetimeIndex, dict of Tet Day1 per year)."""
    holidays = []
    fixed = ['01-01', '04-30', '05-01', '09-02']
    lunar_new_year_day1 = {
        2020: '2020-01-25', 2021: '2021-02-12', 2022: '2022-02-01',
        2023: '2023-01-22', 2024: '2024-02-10', 2025: '2025-01-29',
        2026: '2026-02-17',
    }
    hung_kings = {
        2020: '2020-04-02', 2021: '2021-04-21', 2022: '2022-04-10',
        2023: '2023-04-29', 2024: '2024-04-18', 2025: '2025-04-07',
        2026: '2026-04-26',
    }
    tet_day1_dates = {}
    for year in range(start_year, end_year + 1):
        for mmdd in fixed:
            holidays.append(pd.Timestamp(f'{year}-{mmdd}'))
        if year in lunar_new_year_day1:
            d1 = pd.Timestamp(lunar_new_year_day1[year])
            tet_day1_dates[year] = d1
            shift_days = (d1.dayofweek + 2) % 7
            start_date = d1 - pd.Timedelta(days=shift_days)
            end_date   = start_date + pd.Timedelta(days=8)
            holidays.extend(pd.date_range(start_date, end_date, freq='D').tolist())
        if year in hung_kings:
            holidays.append(pd.Timestamp(hung_kings[year]))
    return pd.DatetimeIndex(sorted(set(holidays))), tet_day1_dates

VN_HOLIDAYS, TET_DAY1_DICT = get_vietnamese_holidays(2020, 2026)
TET_DAY1_LIST = sorted(TET_DAY1_DICT.values())
print(f'Total holiday dates : {len(VN_HOLIDAYS)}')
print(f'Tet Day 1 dates     : {[str(d.date()) for d in TET_DAY1_LIST]}')

---
## ⚙️ Cell 7 — Feature Engineering (v4 Enhanced)

In [ ]:
def build_features(df, vn_holidays, tet_day1_list):
    """
    Build comprehensive features for demand forecasting.
    
    v4 additions vs v3:
    - Yearly lags: lag_357, lag_365, lag_371
    - EWM: ewm_mean_28, ewm_mean_56
    - Tet season: days_to_tet, days_since_tet, is_tet_season
    - Extended holiday windows: pre_14d, post_7d
    - SKU stats: sku_median_qty, sku_cv, sku_active_days
    - Calendar: days_in_month, is_month_start, is_month_end
    """
    df = df.sort_values(['ItemCode', 'Date']).copy()
    
    # 1. Basic Time Features
    df['dayofweek']     = df['Date'].dt.dayofweek.astype('int8')
    df['day']           = df['Date'].dt.day.astype('int8')
    df['month']         = df['Date'].dt.month.astype('int8')
    df['year']          = df['Date'].dt.year.astype('int16')
    df['quarter']       = df['Date'].dt.quarter.astype('int8')
    df['weekofyear']    = df['Date'].dt.isocalendar().week.astype('int8')
    df['days_in_month'] = df['Date'].dt.days_in_month.astype('int8')
    df['is_weekend']    = (df['dayofweek'] >= 5).astype('int8')
    df['is_month_start']= (df['day'] <= 3).astype('int8')
    df['is_month_end']  = (df['day'] >= 28).astype('int8')
    df['is_holiday']    = df['Date'].isin(vn_holidays).astype('int8')
    
    # 2. Holiday Windows (Extended: pre-14d, post-7d)
    holiday_dates_sorted = sorted(vn_holidays)
    unique_dates = df['Date'].unique()
    
    next_hol_map   = {}
    since_hol_map  = {}
    pre14_map      = {}
    pre7_map       = {}
    post7_map      = {}
    post3_map      = {}
    days_to_tet_map    = {}
    days_since_tet_map = {}
    is_tet_season_map  = {}
    
    for d in unique_dates:
        future_hols     = [h for h in holiday_dates_sorted if h > d]
        past_hols       = [h for h in holiday_dates_sorted if h < d]
        days_to_next    = (future_hols[0] - d).days if future_hols else 365
        days_since_last = (d - past_hols[-1]).days  if past_hols  else 365
        
        next_hol_map[d]  = days_to_next
        since_hol_map[d] = days_since_last
        pre14_map[d]  = 1 if (0 < days_to_next  <= 14) else 0
        pre7_map[d]   = 1 if (0 < days_to_next  <= 7)  else 0
        post7_map[d]  = 1 if (0 < days_since_last <= 7) else 0
        post3_map[d]  = 1 if (0 < days_since_last <= 3) else 0
        
        future_tet = [t for t in tet_day1_list if t > d]
        past_tet   = [t for t in tet_day1_list if t <= d]
        d_to_tet   = (future_tet[0] - d).days if future_tet else 365
        d_from_tet = (d - past_tet[-1]).days   if past_tet   else 365
        days_to_tet_map[d]    = min(d_to_tet, 365)
        days_since_tet_map[d] = min(d_from_tet, 365)
        is_tet_season_map[d]  = 1 if (d_to_tet <= 30 or d_from_tet <= 30) else 0
    
    df['days_to_next_holiday']    = df['Date'].map(next_hol_map).astype('int16')
    df['days_since_last_holiday'] = df['Date'].map(since_hol_map).astype('int16')
    df['is_pre_holiday_14d']      = df['Date'].map(pre14_map).astype('int8')
    df['is_pre_holiday_7d']       = df['Date'].map(pre7_map).astype('int8')
    df['is_post_holiday_7d']      = df['Date'].map(post7_map).astype('int8')
    df['is_post_holiday_3d']      = df['Date'].map(post3_map).astype('int8')
    df['days_to_tet']             = df['Date'].map(days_to_tet_map).astype('int16')
    df['days_since_tet']          = df['Date'].map(days_since_tet_map).astype('int16')
    df['is_tet_season']           = df['Date'].map(is_tet_season_map).astype('int8')
    
    # 3. Lag + Rolling Features (all lags >= 56 days for leak-free 56-day horizon)
    grp = df.groupby('ItemCode', observed=True)['Quantity']
    
    # Core lags + yearly seasonality lags
    for lag in [56, 63, 70, 84, 91, 357, 364, 371]:
        df[f'lag_{lag}'] = grp.shift(lag).astype('float32')
    
    # Rolling stats from lag-56 (leak-free)
    lag56 = grp.shift(56)
    for window in [7, 14, 28, 56]:
        df[f'roll_mean_{window}'] = lag56.transform(
            lambda x: x.rolling(window, min_periods=1).mean()
        ).astype('float32')
        df[f'roll_std_{window}'] = lag56.transform(
            lambda x: x.rolling(window, min_periods=1).std()
        ).fillna(0).astype('float32')
    
    # EWM: captures trend with recency bias
    for span in [28, 56]:
        df[f'ewm_mean_{span}'] = lag56.transform(
            lambda x: x.ewm(span=span, min_periods=1).mean()
        ).astype('float32')
    
    # 4. SKU-Level Statistics
    n_days = df['Date'].nunique()
    df['sku_sales_freq']  = grp.transform(lambda x: (x > 0).sum() / n_days).astype('float32')
    df['sku_active_days'] = grp.transform(lambda x: (x > 0).sum()).astype('int16')
    df['sku_median_qty']  = grp.transform('median').astype('float32')
    sku_mean = grp.transform('mean')
    sku_std  = grp.transform('std').fillna(0)
    df['sku_cv'] = (sku_std / (sku_mean + 1e-6)).clip(upper=10).astype('float32')
    
    # 5. Croston: Days-since-last-sale (from lag_56 perspective)
    def days_since_last_sale_transform(series):
        is_zero = (series == 0).astype(int)
        streak = is_zero.groupby((is_zero != is_zero.shift()).cumsum()).cumsum()
        return streak
    df['lag_56_days_since_sale'] = (
        df.groupby('ItemCode', observed=True)['lag_56']
          .transform(days_since_last_sale_transform)
          .astype('int16')
    )
    
    # 6. Price & Margin Features
    if 'SalesAmount' in df.columns:
        df['implied_price'] = np.where(df['Quantity'] > 0, df['SalesAmount'] / df['Quantity'], np.nan)
        df['implied_price'] = df.groupby('ItemCode', observed=True)['implied_price'].ffill()
        sku_median_price = df.groupby('ItemCode', observed=True)['implied_price'].transform('median')
        df['implied_price'] = df['implied_price'].fillna(sku_median_price).fillna(0)
        
        df['implied_cost'] = np.where(df['Quantity'] > 0, df['CostAmount'] / df['Quantity'], np.nan)
        df['implied_cost'] = df.groupby('ItemCode', observed=True)['implied_cost'].ffill()
        sku_median_cost = df.groupby('ItemCode', observed=True)['implied_cost'].transform('median')
        df['implied_cost'] = df['implied_cost'].fillna(sku_median_cost).fillna(0)
        
        df['margin_rate'] = np.where(
            df['implied_price'] > 0,
            (df['implied_price'] - df['implied_cost']) / df['implied_price'],
            0.0
        ).astype('float32')
        df['price_norm'] = np.where(
            sku_median_price > 0, df['implied_price'] / sku_median_price, 1.0
        ).astype('float32')
        
        grp_p = df.groupby('ItemCode', observed=True)
        df['lag_56_price_norm']  = grp_p['price_norm'].shift(56).astype('float32')
        df['lag_56_is_discount'] = (df['lag_56_price_norm'] < 0.95).astype('int8')
        df['lag_56_margin_rate'] = grp_p['margin_rate'].shift(56).astype('float32')
        df.drop(columns=['implied_price', 'implied_cost', 'price_norm', 'margin_rate'], inplace=True)
    
    return df

print('Building v4 enhanced features (may take ~2-3 min) ...')
daily = build_features(daily, VN_HOLIDAYS, TET_DAY1_LIST)
print(f'Total columns after feature engineering: {len(daily.columns)}')
print('Columns:', daily.columns.tolist())


---
## 📐 Cell 8 — WRMSSE Metric

In [ ]:
def compute_wrmsse(y_true, y_pred, train_series, weights):
    """
    WRMSSE = sum(w_i * RMSSE_i)
    RMSSE_i = sqrt(MSE_forecast_i / MSE_naive_i)
    MSE_naive_i = mean((train[t] - train[t-1])^2)
    """
    eps          = 1e-9
    naive_errors = np.diff(train_series, axis=1) ** 2
    denom        = naive_errors.mean(axis=1)
    mse_forecast = ((y_true - y_pred) ** 2).mean(axis=1)
    rmsse        = np.sqrt(mse_forecast / (denom + eps))
    return float(np.sum(weights * rmsse))

print('WRMSSE metric function defined \u2713')

---
## ✂️ Cell 9 — Train/Val Split & Sample Weights

In [ ]:
EXCLUDE_COLS = ['Date', 'Quantity', 'SalesAmount', 'CostAmount']
FEATURE_COLS = [c for c in daily.columns if c not in EXCLUDE_COLS]
print(f'Features ({len(FEATURE_COLS)}): {FEATURE_COLS}')

train_df = daily[daily['Date'] <  SPLIT_DATE].copy()
val_df   = daily[(daily['Date'] >= SPLIT_DATE) & (daily['Date'] <= TRAIN_END)].copy()

# Cross features (leak-free: computed from train only)
print('Computing leak-free cross features ...')
sku_map   = train_df.groupby('ItemCode', observed=True)['Quantity'].mean().to_dict()
dow_map   = train_df.groupby(['ItemCode', 'dayofweek'], observed=True)['Quantity'].mean().to_dict()
month_map = train_df.groupby(['ItemCode', 'month'], observed=True)['Quantity'].mean().to_dict()

def map_cross_features(df):
    df = df.copy()
    df['sku_mean_qty'] = df['ItemCode'].map(sku_map).fillna(0).astype('float32')
    dk = pd.MultiIndex.from_arrays([df['ItemCode'], df['dayofweek']])
    df['sku_dow_mean'] = dk.map(dow_map).fillna(0).astype('float32')
    mk = pd.MultiIndex.from_arrays([df['ItemCode'], df['month']])
    df['sku_month_mean'] = mk.map(month_map).fillna(0).astype('float32')
    return df

daily    = map_cross_features(daily)
train_df = map_cross_features(train_df)
val_df   = map_cross_features(val_df)

if 'sku_mean_qty' not in FEATURE_COLS:
    FEATURE_COLS.extend(['sku_mean_qty', 'sku_dow_mean', 'sku_month_mean'])

# Profit weights (using only pre-split training data)
train_raw = raw[raw['Date'] < SPLIT_DATE].copy()
train_raw['Profit'] = train_raw['SalesAmount'].astype(float) - train_raw['Cost Amount'].astype(float)
sku_profit_val = (
    train_raw.groupby('ItemCode', observed=True)['Profit']
       .sum().reset_index().rename(columns={'Profit': 'total_profit'})
)
sku_profit_val['total_profit'] = sku_profit_val['total_profit'].clip(lower=0)
total_p = sku_profit_val['total_profit'].sum()
sku_profit_val['weight'] = sku_profit_val['total_profit'] / total_p if total_p > 0 else 0.0

# v4 Sample Weights: profit^0.7 / sqrt(denom) — capped at 50x median
train_pivot_denom = train_df.pivot(index='ItemCode', columns='Date', values='Quantity')
naive_errors_w    = np.diff(train_pivot_denom.values, axis=1) ** 2
denom_values_w    = naive_errors_w.mean(axis=1)
denom_df_w        = pd.DataFrame({'ItemCode': train_pivot_denom.index, 'denom': denom_values_w})

sku_stats_val = sku_profit_val.merge(denom_df_w, on='ItemCode', how='left')
for col in ['weight', 'denom', 'total_profit']:
    sku_stats_val[col] = sku_stats_val[col].fillna(0.0)

eps_w = 0.01
# Softer exponent (0.7) gives sparse SKUs more representation than pure profit weighting
sku_stats_val['sample_weight'] = (
    sku_stats_val['total_profit'] ** 0.7 / np.sqrt(sku_stats_val['denom'] + eps_w)
)
pos_mask     = sku_stats_val['sample_weight'] > 0
median_sw    = sku_stats_val.loc[pos_mask, 'sample_weight'].median()
sku_stats_val['sample_weight'] = sku_stats_val['sample_weight'].clip(upper=50 * median_sw)

weight_map    = sku_stats_val.set_index('ItemCode')['sample_weight'].to_dict()
train_weights = train_df['ItemCode'].map(weight_map).fillna(0.0).values
if train_weights.sum() > 0:
    train_weights = train_weights / train_weights.mean()

# Target: square root transform (stabilizes Poisson-like variance)
X_train, y_train = train_df[FEATURE_COLS], np.sqrt(train_df['Quantity'])
X_val,   y_val   = val_df[FEATURE_COLS],   np.sqrt(val_df['Quantity'])

print(f'Train: {len(train_df):,} rows | {train_df["Date"].min().date()} -> {train_df["Date"].max().date()}')
print(f'Val  : {len(val_df):,} rows | {val_df["Date"].min().date()} -> {val_df["Date"].max().date()}')
print(f'Non-zero sample weights: {(train_weights > 0).sum():,} / {len(train_weights):,}')


---
## 🤖 Cell 10 — LightGBM Training (v4 Tuned Params)

In [ ]:
lgb_train = lgb.Dataset(
    X_train, label=y_train, weight=train_weights,
    categorical_feature=['ItemCode'], free_raw_data=False
)
lgb_val = lgb.Dataset(
    X_val, label=y_val,
    categorical_feature=['ItemCode'], reference=lgb_train, free_raw_data=False
)

# v4 Hyperparameters:
# - num_leaves=255 (was 127): more model capacity for complex seasonality
# - lr=0.05 (was 0.03): faster convergence, more diversity with 5000 rounds
# - early_stop=100 (was 50): less aggressive stopping, better generalization
# - min_child_samples=30 (was 20): more conservative splits
# - lambda_l2=0.5 (was 0.1): stronger L2 regularization against overfitting
# - path_smooth=0.5 (new): smooth leaf value estimation for sparse paths
params = {
    'objective':         'regression',
    'metric':            'rmse',
    'learning_rate':     0.05,
    'num_leaves':        255,
    'max_depth':         -1,
    'min_child_samples': 30,
    'feature_fraction':  0.75,
    'bagging_fraction':  0.85,
    'bagging_freq':      1,
    'lambda_l1':         0.05,
    'lambda_l2':         0.5,
    'path_smooth':       0.5,
    'verbose':           -1,
    'seed':              SEED,
    'n_jobs':            -1,
}
callbacks = [lgb.early_stopping(100, verbose=True), lgb.log_evaluation(100)]

print('Training v4 SQRT-Regression LightGBM ...')
model = lgb.train(
    params, lgb_train,
    num_boost_round=5000,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'val'],
    callbacks=callbacks,
)
print(f'Best iteration: {model.best_iteration}')


---
## 📊 Cell 11 — Validate & Compute WRMSSE

In [ ]:
val_preds_sqrt = model.predict(X_val, num_iteration=model.best_iteration)
val_preds      = np.square(np.clip(val_preds_sqrt, 0, None))

val_df = val_df.copy()
val_df['pred'] = val_preds

# Profit-aware post-processing: tiered scaling by profit weight quantiles
profit_weight_map = sku_profit_val.set_index('ItemCode')['weight'].to_dict()
val_df['profit_weight'] = val_df['ItemCode'].map(profit_weight_map).fillna(0.0)
pos_pw = val_df['profit_weight'][val_df['profit_weight'] > 0]
w95 = pos_pw.quantile(0.95)
w80 = pos_pw.quantile(0.80)
w50 = pos_pw.quantile(0.50)

mask_t1 = val_df['profit_weight'] >= w95
mask_t2 = (val_df['profit_weight'] >= w80) & ~mask_t1
mask_t3 = (val_df['profit_weight'] >= w50) & ~mask_t1 & ~mask_t2

val_df.loc[mask_t1, 'pred'] *= 1.20
val_df.loc[mask_t2, 'pred'] *= 1.10
val_df.loc[mask_t3, 'pred'] *= 1.05

# Sparse SKU zeroing (absolute count filter is more robust than relative frequency)
active_days_map = daily.groupby('ItemCode')['sku_active_days'].first().to_dict()
val_df['active_days'] = val_df['ItemCode'].map(active_days_map).fillna(0)
train_sums = train_df.groupby('ItemCode')['Quantity'].sum().to_dict()
val_df['train_sum'] = val_df['ItemCode'].map(train_sums).fillna(0.0)
val_df.loc[val_df['train_sum'] == 0, 'pred'] = 0.0
val_df.loc[val_df['active_days'] < 5, 'pred'] = 0.0

# Compute WRMSSE
y_true_matrix = val_df.pivot(index='ItemCode', columns='Date', values='Quantity').values
y_pred_matrix = val_df.pivot(index='ItemCode', columns='Date', values='pred').values
val_sku_order = val_df.pivot(index='ItemCode', columns='Date', values='Quantity').index
train_pivot   = (
    train_df.pivot(index='ItemCode', columns='Date', values='Quantity')
    .reindex(val_sku_order).fillna(0).values
)
wm_val    = sku_profit_val.set_index('ItemCode')['weight'].to_dict()
val_wts   = np.array([wm_val.get(s, 0.0) for s in val_sku_order])
if val_wts.sum() > 0:
    val_wts /= val_wts.sum()

wrmsse_score = compute_wrmsse(y_true_matrix, y_pred_matrix, train_pivot, val_wts)
print(f'\n🚀 v4 Validation WRMSSE: {wrmsse_score:.4f}')
print(f'   (vs v3 baseline: 0.5585, delta: {wrmsse_score - 0.5585:+.4f})')


---
## 🔄 Cell 12 — Feature Importance

In [ ]:
import matplotlib.pyplot as plt

importance_df = pd.DataFrame({
    'feature':    model.feature_name(),
    'importance': model.feature_importance(importance_type='gain'),
}).sort_values('importance', ascending=False)
print('Top 25 features by gain:')
print(importance_df.head(25).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 10))
top25 = importance_df.head(25)
ax.barh(top25['feature'][::-1], top25['importance'][::-1])
ax.set_xlabel('Importance (Gain)')
ax.set_title('LightGBM v4 Feature Importance — Top 25')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'feature_importance_v4.png', dpi=120)
plt.show()

---
## 🔮 Cell 13 — Forecast Setup

In [ ]:
all_skus  = daily['ItemCode'].cat.categories.tolist()
all_forecast_dates = pd.date_range(VAL_START, EVAL_END, freq='D')
print(f'Forecast: {all_forecast_dates[0].date()} \u2192 {all_forecast_dates[-1].date()} ({len(all_forecast_dates)} days)')
print(f'Total inference rows: {len(all_skus):,} \u00d7 {len(all_forecast_dates)} = {len(all_skus)*len(all_forecast_dates):,}')

---
## 🚀 Cell 14 — 56-Day Inference (Optimized Pipeline)

In [ ]:
print('Generating v4 optimized 56-day forecasts ...')

# Pre-compute full history pivot ONCE outside the loop.
# This eliminates the O(N^2) pivot rebuild that slowed down v3.
# All lag lookups target dates in [TRAIN_END-400d, TRAIN_END], so the pivot
# covers the full necessary window from the start.
HISTORY_START = TRAIN_END - pd.Timedelta(days=400)
history_df    = daily[daily['Date'] >= HISTORY_START][[
    'ItemCode', 'Date', 'Quantity', 'lag_56_price_norm',
    'lag_56_is_discount', 'lag_56_margin_rate', 'lag_56_days_since_sale'
]].copy()
hist_pivot = history_df.pivot(index='ItemCode', columns='Date', values='Quantity').fillna(0)
hist_cols_set = set(hist_pivot.columns)  # fast membership check

holiday_dates_sorted = sorted(VN_HOLIDAYS)
train_sums           = daily.groupby('ItemCode')['Quantity'].sum().to_dict()
active_days_map_inf  = daily.groupby('ItemCode')['sku_active_days'].first().to_dict()
freq_map             = daily.set_index('ItemCode')['sku_sales_freq'].to_dict()

# Cross-feature maps (from full training data, no look-ahead)
train_mask      = daily['Date'] < SPLIT_DATE
sku_map_inf     = daily[train_mask].groupby('ItemCode', observed=True)['Quantity'].mean().to_dict()
dow_map_inf     = daily[train_mask].groupby(['ItemCode', 'dayofweek'], observed=True)['Quantity'].mean().to_dict()
month_map_inf   = daily[train_mask].groupby(['ItemCode', 'month'], observed=True)['Quantity'].mean().to_dict()

# Static per-SKU features
sku_median_map  = daily.groupby('ItemCode')['sku_median_qty'].first().to_dict()
sku_cv_map      = daily.groupby('ItemCode')['sku_cv'].first().to_dict()

# Price/margin/streak from last known historical value
last_price_norm_map  = history_df.groupby('ItemCode')['lag_56_price_norm'].last().to_dict()
last_is_discount_map = history_df.groupby('ItemCode')['lag_56_is_discount'].last().to_dict()
last_margin_rate_map = history_df.groupby('ItemCode')['lag_56_margin_rate'].last().to_dict()
last_streak_map      = history_df.groupby('ItemCode')['lag_56_days_since_sale'].last().to_dict()

# Profit weights for tiered post-processing
profit_wmap_inf = sku_profit_val.set_index('ItemCode')['weight'].to_dict()
all_weights_arr = np.array([profit_wmap_inf.get(s, 0.0) for s in all_skus])
pos_w_arr = all_weights_arr[all_weights_arr > 0]
w95_inf = float(np.quantile(pos_w_arr, 0.95)) if len(pos_w_arr) else 1.0
w80_inf = float(np.quantile(pos_w_arr, 0.80)) if len(pos_w_arr) else 1.0
w50_inf = float(np.quantile(pos_w_arr, 0.50)) if len(pos_w_arr) else 1.0

# Precompute boolean tier masks (same for every forecast day — weights are static)
m_t1_inf = all_weights_arr >= w95_inf
m_t2_inf = (all_weights_arr >= w80_inf) & ~m_t1_inf
m_t3_inf = (all_weights_arr >= w50_inf) & ~m_t1_inf & ~m_t2_inf

all_forecast_records = []

for i, fdate in enumerate(all_forecast_dates):
    if (i + 1) % 10 == 0 or i == 0:
        print(f'  Day {i+1:2d}/56: {fdate.date()}')
    
    # Holiday/Tet proximity for this forecast date
    future_hols          = [h for h in holiday_dates_sorted if h > fdate]
    past_hols            = [h for h in holiday_dates_sorted if h < fdate]
    days_to_next_hol     = (future_hols[0] - fdate).days if future_hols else 365
    days_since_last_hol  = (fdate - past_hols[-1]).days  if past_hols  else 365
    future_tet = [t for t in TET_DAY1_LIST if t > fdate]
    past_tet   = [t for t in TET_DAY1_LIST if t <= fdate]
    d_to_tet   = (future_tet[0] - fdate).days if future_tet else 365
    d_from_tet = (fdate - past_tet[-1]).days   if past_tet   else 365
    
    row_df = pd.DataFrame({'ItemCode': all_skus})
    
    # Lag features (all >= 56 days back, so always in historical data)
    for lag in [56, 63, 70, 84, 91, 357, 364, 371]:
        lag_date = fdate - pd.Timedelta(days=lag)
        if lag_date in hist_cols_set:
            row_df[f'lag_{lag}'] = hist_pivot[lag_date].reindex(all_skus).fillna(0).values
        else:
            row_df[f'lag_{lag}'] = 0.0
    
    # Rolling mean/std from lag-56 window
    lag56_date = fdate - pd.Timedelta(days=56)
    for window in [7, 14, 28, 56]:
        window_start = lag56_date - pd.Timedelta(days=window - 1)
        dates_in_win = [d for d in hist_pivot.columns if window_start <= d <= lag56_date]
        if dates_in_win:
            wv = hist_pivot[dates_in_win].reindex(all_skus).fillna(0)
            row_df[f'roll_mean_{window}'] = wv.mean(axis=1).values
            row_df[f'roll_std_{window}']  = wv.std(axis=1).fillna(0).values
        else:
            row_df[f'roll_mean_{window}'] = row_df[f'roll_std_{window}'] = 0.0
    
    # EWM features
    for span in [28, 56]:
        span_start = lag56_date - pd.Timedelta(days=span * 3)
        dates_ewm  = [d for d in hist_pivot.columns if span_start <= d <= lag56_date]
        if dates_ewm:
            ev = hist_pivot[dates_ewm].reindex(all_skus).fillna(0)
            row_df[f'ewm_mean_{span}'] = ev.apply(
                lambda x: x.ewm(span=span, min_periods=1).mean().iloc[-1], axis=1
            ).values
        else:
            row_df[f'ewm_mean_{span}'] = 0.0
    
    # Static SKU-level features
    row_df['sku_sales_freq']  = row_df['ItemCode'].map(freq_map).fillna(0).values
    row_df['sku_active_days'] = row_df['ItemCode'].map(active_days_map_inf).fillna(0).values
    row_df['sku_median_qty']  = row_df['ItemCode'].map(sku_median_map).fillna(0).values
    row_df['sku_cv']          = row_df['ItemCode'].map(sku_cv_map).fillna(0).values
    
    # Calendar features for this date
    iso_w = fdate.isocalendar()[1]
    time_feats = {
        'dayofweek':               fdate.dayofweek,
        'day':                     fdate.day,
        'month':                   fdate.month,
        'year':                    fdate.year,
        'quarter':                 fdate.quarter,
        'weekofyear':              iso_w,
        'days_in_month':           fdate.days_in_month,
        'is_weekend':              int(fdate.dayofweek >= 5),
        'is_month_start':          int(fdate.day <= 3),
        'is_month_end':            int(fdate.day >= 28),
        'is_holiday':              int(fdate in VN_HOLIDAYS),
        'days_to_next_holiday':    days_to_next_hol,
        'days_since_last_holiday': days_since_last_hol,
        'is_pre_holiday_14d':      1 if (0 < days_to_next_hol  <= 14) else 0,
        'is_pre_holiday_7d':       1 if (0 < days_to_next_hol  <= 7)  else 0,
        'is_post_holiday_7d':      1 if (0 < days_since_last_hol <= 7) else 0,
        'is_post_holiday_3d':      1 if (0 < days_since_last_hol <= 3) else 0,
        'days_to_tet':             min(d_to_tet, 365),
        'days_since_tet':          min(d_from_tet, 365),
        'is_tet_season':           1 if (d_to_tet <= 30 or d_from_tet <= 30) else 0,
    }
    for k, v in time_feats.items():
        row_df[k] = v
    
    # Cross features
    row_df['sku_mean_qty'] = row_df['ItemCode'].map(sku_map_inf).fillna(0).astype('float32')
    dk = pd.MultiIndex.from_arrays([row_df['ItemCode'], row_df['dayofweek']])
    row_df['sku_dow_mean'] = dk.map(dow_map_inf).fillna(0).astype('float32')
    mk = pd.MultiIndex.from_arrays([row_df['ItemCode'], row_df['month']])
    row_df['sku_month_mean'] = mk.map(month_map_inf).fillna(0).astype('float32')
    
    # Price/margin/streak
    row_df['lag_56_price_norm']      = row_df['ItemCode'].map(last_price_norm_map).fillna(1.0).values
    row_df['lag_56_is_discount']     = row_df['ItemCode'].map(last_is_discount_map).fillna(0).values
    row_df['lag_56_margin_rate']     = row_df['ItemCode'].map(last_margin_rate_map).fillna(0).values
    row_df['lag_56_days_since_sale'] = row_df['ItemCode'].map(last_streak_map).fillna(0).values + i
    
    row_df['Date']     = fdate
    row_df['ItemCode'] = row_df['ItemCode'].astype('category')
    
    # Predict (sqrt-space → square back to quantity)
    preds = np.square(np.clip(
        model.predict(row_df[FEATURE_COLS], num_iteration=model.best_iteration), 0, None
    ))
    row_df['forecast'] = preds
    
    # Sparse SKU zeroing
    row_df['train_sum']     = row_df['ItemCode'].map(train_sums).fillna(0.0)
    row_df['active_days_v'] = row_df['ItemCode'].map(active_days_map_inf).fillna(0)
    row_df.loc[row_df['train_sum'] == 0, 'forecast'] = 0.0
    row_df.loc[row_df['active_days_v'] < 5, 'forecast'] = 0.0
    
    # Profit-aware tiered scaling
    row_df.loc[m_t1_inf, 'forecast'] *= 1.20
    row_df.loc[m_t2_inf, 'forecast'] *= 1.10
    row_df.loc[m_t3_inf, 'forecast'] *= 1.05
    
    all_forecast_records.append(row_df[['ItemCode', 'Date', 'forecast']])

forecast_df = pd.concat(all_forecast_records, ignore_index=True)
print(f'Forecast complete: {len(forecast_df):,} rows')


---
## 📝 Cell 15 — Build & Validate Submission

In [ ]:
sample_sub = pd.read_csv(SAMPLE_PATH)
print(f'Sample submission: {sample_sub.shape}')
assert sample_sub.shape[1] == 29, f'Expected 29 cols (id + F1..F28), got {sample_sub.shape[1]}'

F_COLS     = [f'F{i}' for i in range(1, 29)]
val_dates  = pd.date_range(VAL_START,  VAL_END,  freq='D')   # 28 days
eval_dates = pd.date_range(EVAL_START, EVAL_END, freq='D')   # 28 days
assert len(val_dates) == 28 and len(eval_dates) == 28

fc_pivot = forecast_df.pivot(index='ItemCode', columns='Date', values='forecast').fillna(0)

submission_rows = []
for idx_row in sample_sub.itertuples(index=False):
    row_id = idx_row.id
    if '_validation' in row_id:
        sku, dates = row_id.replace('_validation', ''), val_dates
    elif '_evaluation' in row_id:
        sku, dates = row_id.replace('_evaluation', ''), eval_dates
    else:
        raise ValueError(f'Unexpected id format: {row_id}')
    
    preds_28 = np.clip(
        fc_pivot.loc[sku, dates].values if sku in fc_pivot.index else np.zeros(28),
        0, None
    )
    row = {'id': row_id}
    row.update({f'F{i+1}': float(preds_28[i]) for i in range(28)})
    submission_rows.append(row)

submission = pd.DataFrame(submission_rows)

assert len(submission) == len(sample_sub),             'Row count mismatch!'
assert set(submission['id']) == set(sample_sub['id']), 'ID set mismatch!'
assert submission[F_COLS].isna().sum().sum() == 0,     'NaN values found!'
assert (submission[F_COLS] < 0).sum().sum() == 0,      'Negative values found!'
assert submission['id'].duplicated().sum() == 0,        'Duplicate IDs found!'
print('\u2713 All validation checks passed!')

out_path = OUTPUT_DIR / 'submission_v4.csv'
submission.to_csv(out_path, index=False)
flat = submission[F_COLS].values.flatten()
print(f'Saved  \u2192 {out_path}  ({out_path.stat().st_size/1024:.1f} KB)')
print(f'Stats  : mean={flat.mean():.3f}, median={np.median(flat):.3f}, % zero={(flat==0).mean()*100:.1f}%')


---
## Cell 15.5: Post-Processing & Tuning (submission_v4.5)

In [ ]:
# ----------------------------------------------------
# Post-Processing / Tuning (generating submission_v4.5)
# ----------------------------------------------------
print("Tuning predictions for submission_v4.5 ...")

# 1. Compute SKU statistics from in-memory DataFrames
sku_last_sale = daily[daily['Quantity'] > 0].groupby('ItemCode', observed=True)['Date'].max().reset_index(name='last_sale_date')
max_date = daily['Date'].max()
sku_last_sale['days_since_last_sale'] = (max_date - sku_last_sale['last_sale_date']).dt.days

sku_profit_tuned = sku_profit.copy()
sku_profit_tuned['total_profit'] = sku_profit_tuned['total_profit'].clip(lower=0)
total_p = sku_profit_tuned['total_profit'].sum()
sku_profit_tuned['profit_weight'] = sku_profit_tuned['total_profit'] / total_p if total_p > 0 else 0.0

# Merge stats
sku_stats = sku_profit_tuned.merge(sku_last_sale, on='ItemCode', how='left')
sku_stats['days_since_last_sale'] = sku_stats['days_since_last_sale'].fillna(365 * 10)  # very high if no sales at all

# Merge with submission
sub_tuned = submission.copy()
sub_tuned['ItemCode'] = sub_tuned['id'].apply(lambda x: x.replace('_validation', '').replace('_evaluation', ''))
sub_merged = sub_tuned.merge(sku_stats, on='ItemCode', how='left')

# Define tuning masks
mask_stale_1year = sub_merged['days_since_last_sale'] > 365
mask_stale_6months_low_weight = (sub_merged['days_since_last_sale'] > 180) & (sub_merged['profit_weight'] < 0.001)
tuned_mask = mask_stale_1year | mask_stale_6months_low_weight

# Apply zeroing
sub_tuned.loc[tuned_mask, F_COLS] = 0.0

# Cleanup helper columns
sub_tuned = sub_tuned.drop(columns=['ItemCode'])

# 2. Run validations
assert len(sub_tuned) == len(sample_sub),             "Row count mismatch!"
assert set(sub_tuned['id']) == set(sample_sub['id']), "ID set mismatch!"
assert sub_tuned[F_COLS].isna().sum().sum() == 0,     "NaN values found!"
assert (sub_tuned[F_COLS] < 0).sum().sum() == 0,      "Negative values found!"
assert sub_tuned['id'].duplicated().sum() == 0,        "Duplicate IDs found!"
print("All validation checks passed successfully!")

# 3. Save to submission_v4.5.csv
out_path_v4_5 = OUTPUT_DIR / 'submission_v4.5.csv'
sub_tuned.to_csv(out_path_v4_5, index=False)

flat_before = submission[F_COLS].values.flatten()
flat_after = sub_tuned[F_COLS].values.flatten()

print(f"Saved -> {out_path_v4_5} ({out_path_v4_5.stat().st_size/1024:.1f} KB)")
print(f"Stats BEFORE tuning:")
print(f"  mean={flat_before.mean():.6f}, median={np.median(flat_before):.6f}, % zero={(flat_before==0).mean()*100:.2f}%")
print(f"  sum of all forecasts={submission[F_COLS].sum().sum():.2f}")
print(f"Stats AFTER tuning:")
print(f"  mean={flat_after.mean():.6f}, median={np.median(flat_after):.6f}, % zero={(flat_after==0).mean()*100:.2f}%")
print(f"  sum of all forecasts={sub_tuned[F_COLS].sum().sum():.2f}")
print(f"  Forecast sum reduced by: {flat_before.sum() - flat_after.sum():.4f}")


---
## 📋 Cell 16 — Summary

In [ ]:
print('=' * 60)
print('  v3 Validation WRMSSE: 0.5585')
print(f'  v4 Validation WRMSSE: {wrmsse_score:.4f}')
print('=' * 60)
print(f'  Features: {len(FEATURE_COLS)}')
print(f'  Best iter: {model.best_iteration}')
print()
for p in sorted(OUTPUT_DIR.glob('*.csv')) + sorted(OUTPUT_DIR.glob('*.png')):
    print(f'  {p.name}  ({p.stat().st_size/1024:.1f} KB)')